# VTCF — Re-train text_only ablation (fixed)

**What this notebook does:** Re-trains only the broken `text_only` baseline with the fixed code path (`forward_text_only`), then re-evaluates all three models.

**Do NOT re-run** `vision_only` or `full` — those checkpoints are already valid (F1 ≈ 0.996).

## Required Kaggle inputs (Add Data sidebar)
| Dataset | Purpose |
|---|---|
| `vtcf-code` | Upload the `vtcf-code/` folder from this repo as a new dataset |
| `vtcf-bangla-clickbait-frames` | Frame PNGs (already uploaded) |
| `vtcf-checkpoints` *(optional)* | `best_model_vision_only.pt` + `best_model_full.pt` if not already in `/kaggle/working` |

## Settings
- **GPU:** T4
- **Internet:** On

## Success signs during training
- Progress bar shows `contrast=0.0000` (not ~0.746)
- `[text_only debug]` lines appear in epoch 1
- `val_f1` rises toward **~0.98** by epoch 3–5

In [2]:
import glob
import os
import shutil
from pathlib import Path

WORK_DIR = Path("/kaggle/working")
DATA_DIR = WORK_DIR / "data"
MODELS_DIR = WORK_DIR / "models"
SCRIPTS_DIR = WORK_DIR / "scripts"
CKPT_DIR = WORK_DIR / "outputs" / "checkpoints"

for d in [DATA_DIR, MODELS_DIR, SCRIPTS_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

(DATA_DIR / "__init__.py").touch()
(MODELS_DIR / "__init__.py").touch()

# --- Find vtcf-code dataset (upload Kaggle_notebook/vtcf-code/ as a Kaggle dataset) ---
CODE_SRC = None
for candidate in glob.glob("/kaggle/input/**/vtcf-code", recursive=True):
    root = Path(candidate)
    if (root / "config.yaml").exists() and (root / "models" / "fusion_network.py").exists():
        CODE_SRC = root
        break
if CODE_SRC is None:
    for p in glob.glob("/kaggle/input/**/config.yaml", recursive=True):
        root = Path(p).parent
        if (root / "models" / "fusion_network.py").exists():
            CODE_SRC = root
            break
if CODE_SRC is None:
    raise RuntimeError(
        "Could not find vtcf-code dataset. Upload Kaggle_notebook/vtcf-code/ as a Kaggle dataset and add it to this notebook."
    )

print(f"Code source: {CODE_SRC}")

shutil.copy2(CODE_SRC / "config.yaml", WORK_DIR / "config.yaml")
shutil.copy2(CODE_SRC / "data" / "custom_dataset.py", DATA_DIR / "custom_dataset.py")
shutil.copy2(CODE_SRC / "models" / "fusion_network.py", MODELS_DIR / "fusion_network.py")
shutil.copy2(CODE_SRC / "models" / "interpretability.py", MODELS_DIR / "interpretability.py")
shutil.copy2(CODE_SRC / "scripts" / "train.py", SCRIPTS_DIR / "train.py")
shutil.copy2(CODE_SRC / "scripts" / "evaluate.py", SCRIPTS_DIR / "evaluate.py")

# CSV — try vtcf-code bundle, then frames meta dataset
csv_dst = DATA_DIR / "verified_live_videos.csv"
if (CODE_SRC / "data" / "verified_live_videos.csv").exists():
    shutil.copy2(CODE_SRC / "data" / "verified_live_videos.csv", csv_dst)
else:
    csv_candidates = glob.glob(
        "/kaggle/input/**/verified_live_videos.csv", recursive=True
    )
    if not csv_candidates:
        raise RuntimeError("Could not find verified_live_videos.csv in any input dataset.")
    shutil.copy2(csv_candidates[0], csv_dst)

print("✅ Code copied to /kaggle/working")

Code source: /kaggle/input/datasets/anybodyk/vtcf-code/vtcf-code
✅ Code copied to /kaggle/working


In [3]:
import yaml

frames_candidates = glob.glob("/kaggle/input/**/extracted_frames", recursive=True)
if not frames_candidates:
    raise RuntimeError("Could not find extracted_frames. Add vtcf-bangla-clickbait-frames dataset.")

FRAMES_PATH = frames_candidates[0]
print(f"Frames: {FRAMES_PATH} ({len(list(Path(FRAMES_PATH).iterdir()))} video folders)")

# Symlink so CSV frame_dir column resolves correctly
dst = WORK_DIR / "data" / "extracted_frames"
if not dst.exists() and not dst.is_symlink():
    os.symlink(FRAMES_PATH, dst)
    print(f"Symlinked {dst} -> {FRAMES_PATH}")

with open(WORK_DIR / "config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["data"]["frames_dir"] = FRAMES_PATH
cfg["data"]["verified_csv"] = str(csv_dst)
with open(WORK_DIR / "config.yaml", "w") as f:
    yaml.dump(cfg, f)

os.chdir(WORK_DIR)
print("✅ Config ready")
print("  frames_dir:", cfg["data"]["frames_dir"])
print("  verified_csv:", cfg["data"]["verified_csv"])

Frames: /kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames (8047 video folders)
Symlinked /kaggle/working/data/extracted_frames -> /kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames
✅ Config ready
  frames_dir: /kaggle/input/datasets/anybodyk/vtcf-bangla-clickbait-frames/vtcf_extracted_frames/extracted_frames
  verified_csv: /kaggle/working/data/verified_live_videos.csv


In [4]:
# Verify the text_only fix is present
fusion_text = (MODELS_DIR / "fusion_network.py").read_text()
assert "forward_text_only" in fusion_text, "Missing forward_text_only — upload the NEW vtcf-code dataset!"
assert "configure_text_only_training" in fusion_text

train_text = (SCRIPTS_DIR / "train.py").read_text()
assert "run_forward" in train_text and "compute_loss_dict" in train_text

eval_text = (SCRIPTS_DIR / "evaluate.py").read_text()
assert "forward_text_only" in eval_text or "run_forward" in eval_text

print("✅ text_only fix verified in all three files")

✅ text_only fix verified in all three files


In [5]:
# Copy existing vision_only + full checkpoints (needed for evaluate.py)
# Skip if they already exist in /kaggle/working from a previous session.
needed = ["best_model_vision_only.pt", "best_model_full.pt"]
for name in needed:
    dst = CKPT_DIR / name
    if dst.exists():
        print(f"Already have {name} ({dst.stat().st_size / 1e9:.2f} GB)")
        continue
    matches = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not matches:
        print(f"⚠️  {name} not found — upload a vtcf-checkpoints dataset or skip evaluate until you have it")
        continue
    shutil.copy2(matches[0], dst)
    print(f"Copied {name} from {matches[0]}")

print("\nCurrent checkpoints:")
for p in sorted(CKPT_DIR.glob("*.pt")):
    print(f"  {p.name}: {p.stat().st_size / 1e9:.2f} GB")

⚠️  best_model_vision_only.pt not found — upload a vtcf-checkpoints dataset or skip evaluate until you have it
⚠️  best_model_full.pt not found — upload a vtcf-checkpoints dataset or skip evaluate until you have it

Current checkpoints:


In [6]:
# Delete OLD broken text_only checkpoint before re-training
for name in ["best_model_text_only.pt", "last_model_text_only.pt"]:
    p = CKPT_DIR / name
    if p.exists():
        print(f"Deleting {name} ({p.stat().st_size / 1e9:.2f} GB)")
        p.unlink()
    else:
        print(f"Not found (OK): {name}")

Not found (OK): best_model_text_only.pt
Not found (OK): last_model_text_only.pt


In [7]:
!python scripts/train.py --config config.yaml --ablation text_only

Using device: cuda
2026-06-29 21:26:56,852 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-29 21:26:56,868 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sagorsarker/bangla-bert-base/875aa80a42ec196c16bd931ae5d85ad949f58b16/config.json "HTTP/1.1 200 OK"
2026-06-29 21:26:56,885 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sagorsarker/bangla-bert-base/875aa80a42ec196c16bd931ae5d85ad949f58b16/config.json "HTTP/1.1 200 OK"
config.json: 100%|█████████████████████████████| 491/491 [00:00<00:00, 2.49MB/s]
2026-06-29 21:26:56,955 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 21:26:56,955 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


In [8]:
# Free disk — last_* checkpoints are not used by evaluate.py
p = CKPT_DIR / "last_model_text_only.pt"
if p.exists():
    size_gb = p.stat().st_size / 1e9
    p.unlink()
    print(f"Deleted last_model_text_only.pt ({size_gb:.2f} GB)")
else:
    print("No last_model_text_only.pt to delete")

print("\nCheckpoints for evaluation:")
for p in sorted(CKPT_DIR.glob("best_model_*.pt")):
    print(f"  {p.name}: {p.stat().st_size / 1e9:.2f} GB")

Deleted last_model_text_only.pt (2.33 GB)

Checkpoints for evaluation:
  best_model_text_only.pt: 2.33 GB


In [9]:
!python scripts/evaluate.py --config config.yaml

Using device: cuda
2026-06-29 21:38:21,359 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-29 21:38:21,375 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sagorsarker/bangla-bert-base/875aa80a42ec196c16bd931ae5d85ad949f58b16/config.json "HTTP/1.1 200 OK"
2026-06-29 21:38:21,443 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 21:38:21,443 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-29 21:38:21,506 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 21:38:21,569 | INFO | HTTP Request: GET https://huggingface.co/api/models/sagorsarker/bangla-bert-base/tr

In [10]:
# Show updated results
report = Path("/kaggle/working/outputs/kaggle_report/evaluation_report.md")
if report.exists():
    print(report.read_text())
else:
    print("Report not found at", report)

log = Path("/kaggle/working/outputs/logs/training_text_only.csv")
if log.exists():
    print("\n--- training_text_only.csv (last 3 rows) ---")
    lines = log.read_text().strip().splitlines()
    print("\n".join(lines[:1] + lines[-3:]))

Report not found at /kaggle/working/outputs/kaggle_report/evaluation_report.md

--- training_text_only.csv (last 3 rows) ---
epoch,train_loss,det_loss,contrast_loss,val_loss,val_f1,val_accuracy,val_auc,tds_correlation,tds_mean_clickbait,tds_mean_nonclickbait,lr
8,0.003200,0.003200,0.000000,0.054463,0.991302,0.991304,0.999191,,,,0.00000117
9,0.002273,0.002273,0.000000,0.055732,0.990060,0.990062,0.999228,,,,0.00000030
10,0.002176,0.002176,0.000000,0.055896,0.990060,0.990062,0.999241,,,,0.00000000


In [11]:
# Run evaluation (works with just text_only checkpoint)
!python scripts/evaluate.py --config config.yaml

# Read report from correct path
from pathlib import Path
report = Path("/kaggle/working/outputs/evaluation_report.md")
if report.exists():
    print(report.read_text())
else:
    print("Still no report — check evaluate cell output for errors")

Using device: cuda
2026-06-29 21:43:41,379 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-29 21:43:41,395 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sagorsarker/bangla-bert-base/875aa80a42ec196c16bd931ae5d85ad949f58b16/config.json "HTTP/1.1 200 OK"
2026-06-29 21:43:41,465 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 21:43:41,529 | INFO | HTTP Request: HEAD https://huggingface.co/sagorsarker/bangla-bert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-06-29 21:43:41,594 | INFO | HTTP Request: GET https://huggingface.co/api/models/sagorsarker/bangla-bert-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-06-29 21:43:41,660 | INFO | HTTP Request: GET https://huggingface.co/api/models/s